# Kaggle — dataset redimensionado `melanoma-isic2020-512` (F2.0)

Corre **una sola vez**. Antes de correr:

1. **Add Data → Competitions → `SIIM-ISIC Melanoma Classification`** (imágenes originales, `jpeg/train/`).
2. **Add Data → Your Datasets → `melanoma-isic2020-splits`** (manifiesto con `sha256_resized`).
3. Sin GPU (es trabajo de CPU) y con Internet activado (clona el repositorio).

Qué hace: clona el repositorio en un commit fijo, lee `resize.long_side` y `resize.jpeg_quality` de
`configs/data/isic2020.yaml` (no se escriben a mano), redimensiona las 33,126 imágenes con
`melanoma.data.resize.resize_many` (el mismo código de F1), calcula el SHA256 de cada resultado y lo
compara con `sha256_resized` del manifiesto local. **Reporta cuántas coinciden.** Si no coinciden
todas no es un fallo: se documenta la diferencia y su causa probable (versión de Pillow/libjpeg).

Al final deja `{WORK}/isic2020_512` listo (imágenes + `dataset-metadata.json` +
`resize_verification.json`) y el dataset se crea **desde la interfaz**: *Save Version → Save & Run
All* y, en la pestaña *Output* de la versión, *New Dataset*. No se usa la API de Kaggle: subir
33,126 archivos uno por uno desde un kernel de Kaggle hacia Kaggle falló dos veces con 500/503
(2026-09-17); la ruta por interfaz no transfiere nada porque los archivos ya están del lado de Kaggle.

Verificado en local con `make check-notebooks` (sandbox con la misma estructura que Kaggle).


In [ ]:
# Parámetros. REPO_SHA: commit del repositorio que se ejecuta (no `main`).
REPO_URL = "https://github.com/Edgar-Ontiveros/melanoma-triage.git"
REPO_SHA = "main"  # ← sustituir por el SHA de la corrida
COMPETITION_DIR = "/kaggle/input/competitions/siim-isic-melanoma-classification/jpeg/train"
SPLITS_DIR = "/kaggle/input/datasets/edgaronti26/melanoma-isic2020-splits"
WORK = "/kaggle/working"
# Fuera de /kaggle/working: la API no puede listar una salida de 33 mil archivos (429, verificado
# el 2026-09-17), y el dataset se publica desde la laptop (scripts/kaggle_run.py dataset-images).
# Para crear el dataset desde la interfaz (Output → New Dataset) usar f"{WORK}/isic2020_512".
OUT_DIR = "/tmp/isic2020_512"
DATASET_ID = "edgaronti26/melanoma-isic2020-512"
WORKERS = 4

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_DIR = f"{WORK}/repo"
subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_SHA], check=True)
os.chdir(REPO_DIR)
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("commit", SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
# El .pth de la instalación editable solo se lee al arrancar el intérprete: este kernel ya está
# corriendo y no vería src/. Sin esto, `import melanoma` resolvía al directorio del clon como
# paquete namespace y `melanoma.data` era la carpeta data/ del repo (F2, 2026-09-17).
sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import PIL
from omegaconf import OmegaConf

from melanoma.data.resize import resize_many

cfg = OmegaConf.load("configs/data/isic2020.yaml")
LONG_SIDE, QUALITY = int(cfg.resize.long_side), int(cfg.resize.jpeg_quality)
print(f"Pillow {PIL.__version__} · long_side={LONG_SIDE} · jpeg_quality={QUALITY}")

manifest = pd.read_csv(f"{SPLITS_DIR}/isic2020.csv", dtype={"image_id": str, "patient_id": str})
assert "sha256_resized" in manifest.columns, "el manifiesto no tiene sha256_resized"
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
pairs = [(Path(COMPETITION_DIR) / f"{i}.jpg", out / f"{i}.jpg") for i in manifest["image_id"]]
missing = [s for s, _ in pairs if not s.exists()]
assert not missing, f"faltan {len(missing)} originales en la competencia: {missing[:5]}"

t0 = time.time()
results = resize_many(pairs, long_side=LONG_SIDE, quality=QUALITY, workers=WORKERS)
print(f"{len(results)} imágenes redimensionadas en {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Verificación contra las huellas locales (F2.0, pasos 4-5)
manifest["sha256_kaggle"] = [r["sha256"] for r in results]
manifest["width_kaggle"] = [r["width"] for r in results]
manifest["height_kaggle"] = [r["height"] for r in results]
match = manifest["sha256_kaggle"] == manifest["sha256_resized"]
same_dims = (manifest["width_kaggle"] == manifest["width_resized"]) & (
    manifest["height_kaggle"] == manifest["height_resized"]
)
n = len(manifest)
report = {
    "pillow_kaggle": PIL.__version__,
    "long_side": LONG_SIDE,
    "jpeg_quality": QUALITY,
    "images": n,
    "sha256_match": int(match.sum()),
    "sha256_match_pct": round(100 * match.mean(), 2),
    "dims_match": int(same_dims.sum()),
}
print(json.dumps(report, indent=2))
Path(WORK, "resize_verification.json").write_text(json.dumps(report, indent=2))
cols = ["image_id", "sha256_resized", "sha256_kaggle", "width_resized", "width_kaggle"]
manifest.loc[~match, cols].head(20)

## Interpretación

- **Todas coinciden:** la copia de Kaggle es byte-idéntica a `data/processed/isic2020_512` de la laptop.
- **No todas coinciden pero las dimensiones sí:** el redimensionado es el mismo (mismo código, mismos
  parámetros); difiere la codificación JPEG por versión de Pillow/libjpeg. La procedencia sigue siendo
  válida porque F1 verificó los **originales** byte a byte (200/200). Anotar el porcentaje, la versión
  de Pillow de Kaggle y la local (12.3.0 el 2026-09-17) en `docs/DATA.md` y en `docs/specs/F2.md`.
- **Dimensiones distintas:** algo cambió en el código o en los parámetros. Detener y revisar el SHA.


In [ ]:
# Dejar la salida lista para crear el dataset desde la interfaz. No se usa `kaggle datasets
# create`: subir 33 mil archivos uno por uno desde Kaggle hacia Kaggle falló con 500/503.
meta = {
    "title": "melanoma-isic2020-512",
    "id": DATASET_ID,
    "licenses": [{"name": "CC-BY-NC-SA-4.0"}],
    "subtitle": "ISIC 2020 redimensionado a 512 px (JPEG q95) para el proyecto melanoma",
    "description": (
        "Imagenes del SIIM-ISIC 2020 Challenge Dataset (CC-BY-NC 4.0, "
        f"https://doi.org/10.34970/2020-ds01) redimensionadas al lado largo de {LONG_SIDE} px "
        f"con JPEG calidad {QUALITY}, con el codigo de {REPO_URL} (commit {SHA}). "
        f"Verificacion de huellas: {report['sha256_match']}/{n} coinciden con la copia local."
    ),
    "keywords": ["medicine", "image"],
}
Path(OUT_DIR, "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
Path(OUT_DIR, "resize_verification.json").write_text(json.dumps(report, indent=2))
Path(WORK, "resize_verification.json").write_text(json.dumps(report, indent=2))
Path(WORK, "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
n_files = sum(1 for _ in Path(OUT_DIR).glob("*.jpg"))
print(
    f"""
{n_files} JPEG listos en {OUT_DIR} con dataset-metadata.json y resize_verification.json.

PARA CREAR EL DATASET (sin transferir nada; los archivos ya están en Kaggle):
  1. Save Version → Save & Run All (esperar a que termine).
  2. Abrir la versión → pestaña Output → carpeta isic2020_512 → "New Dataset".
  3. Título: melanoma-isic2020-512, visibilidad privada. Debe quedar como {DATASET_ID}.
  4. Copiar la salida de la celda de verificación a docs/DATA.md.
Los notebooks de entrenamiento lo adjuntan en /kaggle/input/datasets/{DATASET_ID}/.
"""
)